# Session 1: height from a shadow and from one angle
**A. Real-world problem.** You cannot climb a tower or a tree with a tape measure. But you can
stand on the ground, look up at the top, and read the angle on a phone. Two ideas turn that
angle (or a shadow) into a height: *similar triangles* and the *tangent*.

**Plan.** (1) shadow method, (2) tangent method, (3) add your eye height, (4) plot height
against angle.

In [ ]:
import pathlib
import math

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
EYE = 1.6   # metres: a typical eye height above the ground (an assumption you can change)

## Step 1: the shadow method (similar triangles)
At the same moment the sun makes the same angle with the ground for everything. A stick and a
tower then form two triangles with the same shape, so their height-to-shadow ratios match:

    tower_height / tower_shadow = stick_height / stick_shadow

In [ ]:
def shadow_height(stick_height, stick_shadow, tower_shadow):
    """Height (m) of a tower from the shadow of a reference stick measured at the same moment."""
    if stick_shadow <= 0:
        raise ValueError("the stick shadow must be positive")
    return stick_height * tower_shadow / stick_shadow

## Step 2: the angle of elevation
Stand a distance `d` from the base. The angle `a` is measured from the horizontal up to the top.
In the right triangle, tan(a) = (height above your eye) / d, so

    height above eye = d * tan(a)        total height = d * tan(a) + eye height

In [ ]:
def height_from_angle(d, angle_deg, eye=EYE):
    """Total height (m) from distance d (m, horizontal, to the base) and angle of elevation (degrees)."""
    if not 0 < angle_deg < 90:
        raise ValueError("angle of elevation must be between 0 and 90 degrees")
    return d * math.tan(math.radians(angle_deg)) + eye


def angle_for_height(d, height, eye=EYE):
    """The angle (degrees) you should read if the tower has this total height."""
    return math.degrees(math.atan2(height - eye, d))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("shadow method: stick 1.5 m, shadow 1.0 m; tower shadow 18 m ->", shadow_height(1.5, 1.0, 18.0), "m")
    print("angle method: d = 40 m, a = 35 degrees ->", round(height_from_angle(40, 35), 2), "m")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([0, 18], [0, 0], "k-")
    ax.plot([18, 18], [0, 27], color="tab:blue", lw=4, label="tower: 27 m, shadow 18 m")
    ax.plot([18, 0], [27, 0], "--", color="orange")
    ax.plot([0, 0], [0, 1.5], color="tab:red", lw=4, label="stick: 1.5 m, shadow 1.0 m")
    ax.plot([0, 1.0], [1.5, 0], "--", color="orange")
    ax.set_aspect("equal")
    ax.set_xlabel("distance along the ground (m)")
    ax.set_ylabel("height (m)")
    ax.set_title("Similar triangles: same sun angle, same ratio")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "shadow_triangles.png", dpi=120)

    a = np.linspace(1, 80, 200)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for d in (20, 40, 60):
        ax2.plot(a, d * np.tan(np.radians(a)) + EYE, label=f"d = {d} m")
    ax2.axhline(30, color="gray", ls=":")
    ax2.set_ylim(0, 100)
    ax2.set_xlabel("angle of elevation (degrees)")
    ax2.set_ylabel("total height (m)")
    ax2.set_title("Height = d tan(angle) + 1.6 m")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "height_vs_angle.png", dpi=120)

## Where this stops
One angle needs the distance `d` to the base. If a fence or a river stops you reaching the
base, Session 2 uses two angles instead.

## Coding tasks
1. **Easy.** Change the angle in `lesson.py` to 25 and 50 degrees at d = 40 m; predict each height first, then compare with the printed value.
2. **Stretch.** Write `distance_for_angle(height, angle, eye)` that gives the distance you should stand at to read a given angle for a tower of a known height, and test it on the 10.4 m example from Section F.
3. **Explain.** Explain why the stick and the tower must have their shadows measured at the same moment, using the word similar.

## Assessment
- **A. Conceptual.** Why do a stick and a tower form similar triangles with their shadows, and what would go wrong if you measured the stick's shadow at 9 am and the tower's at 11 am?
- **B. Numerical.** A tree casts a 15 m shadow when a 2.0 m stick casts a 1.2 m shadow. How tall is the tree? Then a different tree is seen at 40 degrees from 12 m away with the eye at 1.6 m; how tall is it?
- **C. Graph and simulation interpretation.** On the graph of total height against angle for d = 40 m, read the heights at 20, 45 and 60 degrees. What happens to the height per degree as the angle grows?
- **D. Design and optimisation challenge.** Design: you think a tree is about 12 m tall and you have a 50 m tape. Choose where to stand and justify it. What would be wrong with standing 2 m away, or 45 m away?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: stick 1.5 m with shadow 1.0 m, tower shadow 18 m gives 27 m",
                   abs(L.shadow_height(1.5, 1.0, 18.0) - 27.0) < 1e-9))
    h = L.height_from_angle(40, 35, 1.6)
    checks.append((f"worked example: d = 40 m, 35 degrees, eye 1.6 m gives 29.61 m (got {h:.2f})", abs(h - 29.608) < 0.01))
    checks.append(("hand calculation: 45 degrees at 30 m with zero eye height is 30 m", abs(L.height_from_angle(30, 45, 0.0) - 30.0) < 1e-9))
    checks.append(("hand calculation: 30 degrees at 20 m, no eye, is 20/sqrt(3) = 11.547 m",
                   abs(L.height_from_angle(20, 30, 0.0) - 20 / math.sqrt(3)) < 1e-9))
    a = L.angle_for_height(40, 29.608, 1.6)
    checks.append(("angle_for_height undoes height_from_angle", abs(a - 35.0) < 0.01))
    checks.append(("doubling the distance doubles the height above the eye",
                   abs((L.height_from_angle(80, 20, 0) - 2 * L.height_from_angle(40, 20, 0))) < 1e-9))
    for bad in (0, 90, -5, 120):
        try:
            L.height_from_angle(10, bad)
            ok = False
        except ValueError:
            ok = True
        checks.append((f"edge: angle {bad} degrees is rejected", ok))
    try:
        L.shadow_height(1.5, 0.0, 10.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("edge: zero stick shadow is rejected", ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
